# Dual-Model Qwen3.8-27B + LangGraph Gateway for VS Code Agents

Architecture optimized for **VS Code coding agents** (Cline, Roo Code, Continue, Kilo Code, OpenCode) on Kaggle:

| Component | Model | Hardware | Port | Role |
| :--- | :--- | :--- | :--- | :--- |
| **Solver** | `Qwen3.8-27B` | **GPU (T4 x2)** (`-ngl 99`, `-np 1`, `32K` ctx) | `:8000` | Primary coding, planning, and tool-calling model |
| **Compactor / Router** | `Qwen3-1.7B-Q8_0` | **GPU 1** (`-ngl 99`, `8K` ctx) or CPU | `:8001` | Background pre-compaction + lightweight title/summary offloading |
| **LangGraph Gateway** | `StateGraph` | Python / FastAPI | `:8080` | Tool pruning + file/error ledger + async pre-compaction + dynamic sampling + thinking-loop guard + `/metrics` |

### Recommended Kaggle Settings
- **Accelerator:** **GPU T4 x2** (32 GB VRAM)
- **Internet:** **ON**

## 1. Configuration & Hardware Check

In [ ]:
import os, shutil, subprocess

# Main Solver (GPU)
SOLVER_REPO = "unsloth/Qwen3.8-27B-GGUF"
SOLVER_FILE = "Qwen3.8-27B-UD-IQ4_XS.gguf"   # ~14.3 GB (fast, leaves VRAM for 32K ctx)
USE_MTP_SPECULATIVE = False
SOLVER_CTX = 32768

# Dedicated Context Compactor (GPU 1 on T4 x2, or CPU if single 16GB GPU is tight)
COMPACTOR_REPO = "Qwen/Qwen3-1.7B-GGUF"
COMPACTOR_FILE = "Qwen3-1.7B-Q8_0.gguf"      # ~1.83 GB
COMPACTOR_CTX = 8192
COMPACTOR_ON_GPU = True

LOCAL_SOLVER_PATH = ""  # Optional: "/kaggle/input/.../model.gguf"

SOLVER_PORT = 8000
COMPACTOR_PORT = 8001
GATEWAY_PORT = 8080
BIN_DIR = "/tmp/llama-bin"

candidates = [p for p in ["/tmp", "/kaggle/working"] if os.path.exists(p)]
STORAGE_ROOT = max(candidates, key=lambda p: shutil.disk_usage(p).free)
MODELS_DIR = os.path.join(STORAGE_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(BIN_DIR, exist_ok=True)

for p in candidates:
    print(f"Free disk at {p}: {shutil.disk_usage(p).free / (1024**3):.1f} GB")

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
HAS_GPU = gpu_check.returncode == 0
GPU_COUNT = len(gpu_check.stdout.strip().splitlines()) if HAS_GPU else 0
print(f"GPUs ({GPU_COUNT}):", gpu_check.stdout.strip() if HAS_GPU else "CPU-only")

## 2. Download Prebuilt `llama.cpp` (`llama-server`)

In [ ]:
import json, urllib.request, tarfile

req = urllib.request.Request(
    "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=20",
    headers={"User-Agent": "kaggle-notebook"}
)
with urllib.request.urlopen(req) as resp:
    releases = json.loads(resp.read().decode())

if HAS_GPU:
    def needed_for(tag):
        return [
            f"llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
            f"cudart-llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
        ]
else:
    def needed_for(tag):
        return [f"llama-{tag}-bin-ubuntu-x64.tar.gz"]

release = next(
    (
        r for r in releases
        if r.get("tag_name", "").startswith("b")
        and all(
            any(a["name"] == fn for a in r.get("assets", []))
            for fn in needed_for(r["tag_name"])
        )
    ),
    None,
)
if not release:
    raise RuntimeError("No recent llama.cpp release with matching prebuilt assets found.")

tag = release["tag_name"]
assets = {a["name"]: a["browser_download_url"] for a in release["assets"]}
print("Using llama.cpp release:", tag)

for fname in needed_for(tag):
    url = assets[fname]
    tgz_path = f"/tmp/{fname}"
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(url, tgz_path)
    with tarfile.open(tgz_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = member.name.split("/", 1)
            if len(parts) < 2 or not parts[1]:
                continue
            member.name = parts[1]
            tf.extract(member, BIN_DIR)
    os.remove(tgz_path)

LLAMA_SERVER = os.path.join(BIN_DIR, "llama-server")
os.chmod(LLAMA_SERVER, 0o755)
print("Ready:", LLAMA_SERVER)

## 3. Download Solver & Compactor Models + Install LangGraph

In [ ]:
!pip install -qU huggingface_hub langgraph langchain-core httpx fastapi uvicorn openai
from huggingface_hub import hf_hub_download

if LOCAL_SOLVER_PATH and os.path.exists(LOCAL_SOLVER_PATH):
    SOLVER_MODEL_PATH = LOCAL_SOLVER_PATH
else:
    SOLVER_MODEL_PATH = hf_hub_download(repo_id=SOLVER_REPO, filename=SOLVER_FILE, local_dir=MODELS_DIR)

COMPACTOR_MODEL_PATH = hf_hub_download(repo_id=COMPACTOR_REPO, filename=COMPACTOR_FILE, local_dir=MODELS_DIR)

print("Solver model:", SOLVER_MODEL_PATH)
print("Compactor model:", COMPACTOR_MODEL_PATH)

## 4. Start Both `llama-server` Instances
- **Solver (`:8000`)**: GPU (`-ngl 99`), single slot (`-np 1`), `32K` context with quantized KV cache (`q8_0`), Flash Attention, `--jinja`.
- **Compactor (`:8001`)**: CPU (`-ngl 0`), single slot (`-np 1`), `8K` context (zero VRAM footprint).

In [ ]:
import time, urllib.request

subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(0.5)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{BIN_DIR}:{env.get('LD_LIBRARY_PATH', '')}"

solver_cmd = [
    LLAMA_SERVER,
    "-m", SOLVER_MODEL_PATH,
    "--host", "127.0.0.1",
    "--port", str(SOLVER_PORT),
    "-c", str(SOLVER_CTX),
    "-np", "1",
    "-ngl", "99" if HAS_GPU else "0",
    "-fa", "on",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "--jinja",
    "--alias", "qwen3.8-27b",
]
if GPU_COUNT >= 2 and COMPACTOR_ON_GPU:
    solver_cmd += ["--tensor-split", "0.58,0.42"]
if USE_MTP_SPECULATIVE:
    solver_cmd += ["--spec-type", "draft-mtp", "--spec-draft-p-min", "0.35", "--spec-draft-n-max", "2"]

compactor_env = env.copy()
if GPU_COUNT >= 2 and COMPACTOR_ON_GPU:
    compactor_env["CUDA_VISIBLE_DEVICES"] = "1"

compactor_ngl = "99" if (HAS_GPU and COMPACTOR_ON_GPU) else "0"
compactor_cmd = [
    LLAMA_SERVER,
    "-m", COMPACTOR_MODEL_PATH,
    "--host", "127.0.0.1",
    "--port", str(COMPACTOR_PORT),
    "-c", str(COMPACTOR_CTX),
    "-np", "1",
    "-ngl", compactor_ngl,
    "-fa", "on" if compactor_ngl != "0" else "off",
    "--jinja",
    "--alias", "qwen3-1.7b-compactor",
]

# start_new_session=True isolates background processes from Jupyter SIGINT when stopping other cells
solver_proc = subprocess.Popen(
    solver_cmd,
    stdin=subprocess.DEVNULL,
    stdout=open("/tmp/solver.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
compactor_proc = subprocess.Popen(
    compactor_cmd,
    stdin=subprocess.DEVNULL,
    stdout=open("/tmp/compactor.log", "w"),
    stderr=subprocess.STDOUT,
    env=compactor_env,
    start_new_session=True,
)

def wait_health(proc, port, log_file, name):
    for i in range(180):
        if proc.poll() is not None:
            print(open(log_file).read())
            raise RuntimeError(f"{name} exited with code {proc.returncode}")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health") as r:
                if r.status == 200:
                    print(f"{name} ready in {i}s")
                    return
        except Exception:
            time.sleep(1)
    print(open(log_file).read())
    raise TimeoutError(f"Timed out waiting for {name}")

wait_health(compactor_proc, COMPACTOR_PORT, "/tmp/compactor.log", "Compactor (:8001)")
wait_health(solver_proc, SOLVER_PORT, "/tmp/solver.log", "Solver (:8000)")

## 5. Start the LangGraph Gateway (`:8080`)
Pipeline in `scripts/langgraph_vscode_gateway.py` (from the cloned repository):
1. **`prune_tool_outputs`:** Compresses bulky older tool logs (>3 turns old) while preserving first/last 30 lines + any error/traceback lines.
2. **`compact_context` (Soft 60% -> Hard 75% -> Target 40%):** Triggers **async background pre-compaction** on `:8001` at 60% so summaries are already cached in SQLite by 75% (0 ms wait), plus pins a deterministic **File & Error Ledger** so paths/line numbers never drift.
3. **`analyze_and_route`:** Detects errors/tracebacks and stuck tool loops, adjusts **dynamic sampling** (`temp=0.6` + thinking ON for reasoning/debugging; `temp=0.15` + thinking OFF for routine tool continuations), and offloads lightweight title/summary requests to `:8001`.
4. **Streaming & Non-Streaming Guards:** Aborts intra-generation **thinking loops** in real time and auto-retries with `enable_thinking=False`, plus repairs malformed JSON tool arguments and exposes **`GET /metrics`**.

In [ ]:
import os, subprocess

PROJECT_DIR = "/kaggle/working/langgraph-optimized-kaggle-llm"
PROJECT_GIT_DIR = os.path.join(PROJECT_DIR, ".git")
if os.path.isdir(PROJECT_GIT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/binief/langgraph-optimized-kaggle-llm.git",
        PROJECT_DIR,
    ], check=True)
SCRIPTS_DIR = os.path.join(PROJECT_DIR, "scripts")
print("Project scripts:", SCRIPTS_DIR)

In [ ]:
import subprocess, time, urllib.request

subprocess.run(["pkill", "-f", "uvicorn langgraph_vscode_gateway:app"], check=False)
time.sleep(0.5)

gw_env = os.environ.copy()
gw_env["SOLVER_BASE"] = f"http://127.0.0.1:{SOLVER_PORT}/v1"
gw_env["COMPACTOR_BASE"] = f"http://127.0.0.1:{COMPACTOR_PORT}/v1"

gw_proc = subprocess.Popen(
    [
        "uvicorn", "langgraph_vscode_gateway:app",
        "--host", "0.0.0.0",
        "--port", str(GATEWAY_PORT),
        "--timeout-keep-alive", "300",
    ],
    cwd=SCRIPTS_DIR,
    stdin=subprocess.DEVNULL,
    stdout=open("/tmp/gateway.log", "w"),
    stderr=subprocess.STDOUT,
    env=gw_env,
    start_new_session=True,
)

for _ in range(20):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{GATEWAY_PORT}/health") as r:
            print("Gateway healthy:", r.read().decode())
            break
    except Exception:
        time.sleep(0.5)

## 6. Test Solver + Compactor End-to-End
Tests:
1. Streaming response from `qwen3.8-27b`.
2. Compactor summarization via `:8001` by sending an oversized synthetic history.

In [ ]:
from openai import OpenAI
import httpx

client = OpenAI(base_url=f"http://127.0.0.1:{GATEWAY_PORT}/v1", api_key="not-needed")

print("=== 1. Streaming test ===")
stream = client.chat.completions.create(
    model="qwen3.8-27b",
    messages=[{"role": "user", "content": "Write a 5-line Python LRU cache decorator example."}],
    stream=True,
    max_tokens=200,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

print("\n\n=== 2. Compactor test (:8001) ===")
r = httpx.post(
    f"http://127.0.0.1:{COMPACTOR_PORT}/v1/chat/completions",
    json={
        "model": "qwen3-1.7b-compactor",
        "messages": [{"role": "user", "content": "Summarize in 1 bullet: edited auth.py to fix JWT expiry check and pytest passed."}],
        "max_tokens": 80,
        "chat_template_kwargs": {"enable_thinking": False},
    },
    timeout=60,
)
print(r.json()["choices"][0]["message"]["content"])

## 7. Expose Publicly via Cloudflare Tunnel (for VS Code)
Point your VS Code extension (Cline, Roo Code, Continue, Kilo Code, etc.) to:
- **Provider:** `OpenAI Compatible`
- **Base URL:** `https://<your-tunnel>.trycloudflare.com/v1`
- **API Key:** `not-needed`
- **Model ID:** `qwen3.8-27b`

In [ ]:
%%bash
if [ ! -f /usr/local/bin/cloudflared ]; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi

pkill -f "cloudflared tunnel" || true
nohup cloudflared tunnel --url http://localhost:8080 > /tmp/tunnel.log 2>&1 < /dev/null &
sleep 5
echo "Public Base URL (add /v1 in VS Code):"
grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /tmp/tunnel.log | head -n 1

## 8. Check Logs Anytime (Non-blocking)

In [ ]:
!curl -s http://127.0.0.1:8080/metrics && echo '' && tail -n 25 /tmp/gateway.log /tmp/solver.log /tmp/compactor.log